# mms-300m karaoke finetune

In [ ]:
import json
from typing import cast

import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import login as hf_login
from transformers import (
    EarlyStoppingCallback,
    EvalPrediction,
    Trainer,
    TrainingArguments,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
)
from transformers.modeling_outputs import CausalLMOutput

In [ ]:
hf_login()

In [ ]:
dataset = load_dataset("NextFire/karaoke-alignments", split="train")

In [ ]:
max_duration = 120

dataset = dataset.filter(
    lambda x: x.metadata.duration_seconds <= max_duration,
    input_columns=["audio"],
    new_fingerprint="mms-300m_filtered_dataset",
)

In [ ]:
vocab = dataset.map(
    lambda batch: {
        "vocab": list(
            set(
                letter
                for sample in batch
                for mora in sample
                for letter in mora["value"]
            )
        )
    },
    input_columns=["morae"],
    remove_columns=dataset.column_names,
    batched=True,
    batch_size=None,
    keep_in_memory=True,
)["vocab"]

vocab_dict = {v: k for k, v in enumerate(sorted(vocab))}
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

with open("vocab.json", "w") as f:
    json.dump(vocab_dict, f)

In [ ]:
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    "./",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

In [ ]:
class Wav2Vec2ForForcedAligner(Wav2Vec2ForCTC):
    IGNORE_LOSS_INDEX = -100

    def forward(
        self,
        input_values: torch.Tensor | None,
        attention_mask: torch.Tensor | None = None,
        output_attentions: bool | None = None,
        output_hidden_states: bool | None = None,
        return_dict: bool | None = None,
        labels: torch.Tensor | None = None,
        **kwargs,
    ):
        outputs = super().forward(
            input_values=input_values,
            attention_mask=attention_mask,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
            labels=None,  # skip CTC loss
            **kwargs,
        )
        outputs = cast(CausalLMOutput, outputs)
        loss = None
        if labels is not None:
            assert outputs.logits is not None
            loss = F.cross_entropy(
                outputs.logits.transpose(1, 2),
                labels,
                ignore_index=self.IGNORE_LOSS_INDEX,
                label_smoothing=0.05,
            )
        return CausalLMOutput(
            loss=loss,  # pyright: ignore[reportArgumentType]
            logits=outputs.logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )


model = Wav2Vec2ForForcedAligner.from_pretrained(
    "facebook/mms-300m",
    # attention_dropout=0.0,
    # hidden_dropout=0.0,
    # feat_proj_dropout=0.0,
    # layerdrop=0.0,
    pad_token_id=tokenizer.pad_token_id,
    vocab_size=len(tokenizer),
    ignore_mismatched_sizes=True,
)

model.freeze_feature_encoder()

adapter_stride = model.config.adapter_stride if model.config.add_adapter else 1
ms_per_frame = float(
    1
    / model._get_feat_extract_output_lengths(feature_extractor.sampling_rate)
    * adapter_stride
    * 1000
)

In [ ]:
def prepare_dataset(example):
    audio = example["audio"]
    input_values = processor(
        audio=audio["array"],
        sampling_rate=audio["sampling_rate"],  # pyright: ignore[reportCallIssue]
    ).input_values[0]
    example["input_values"] = input_values

    output_length = int(model._get_feat_extract_output_lengths(len(input_values)))
    labels = [model.IGNORE_LOSS_INDEX] * output_length
    for mora in example["morae"]:
        characters = list(mora["value"])
        start_frame = round(mora["start"] / ms_per_frame)
        assert start_frame < output_length
        end_frame = round(mora["end"] / ms_per_frame)
        assert start_frame <= end_frame <= output_length, (
            f"{start_frame} <= {end_frame} <= {output_length}"
        )
        for frame in range(start_frame, end_frame):
            assert labels[frame] == model.IGNORE_LOSS_INDEX
            position = (frame - start_frame) / (end_frame - start_frame)
            char_idx = int(position * len(characters))
            labels[frame] = tokenizer.convert_tokens_to_ids(characters[char_idx])
    example["labels"] = labels

    return example


dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset.column_names,
    new_fingerprint="mms-300m_prepared_dataset",
)
split = dataset.train_test_split(test_size=0.1, seed=42)

In [ ]:
def data_collator(examples) -> dict[str, torch.Tensor]:
    input_features = [{"input_values": e["input_values"]} for e in examples]
    labels = [{"input_ids": e["labels"]} for e in examples]
    batch = processor.pad(input_features, padding=True, return_tensors="pt")
    labels_batch = processor.pad(labels=labels, padding=True, return_tensors="pt")
    batch["labels"] = labels_batch["input_ids"].masked_fill(
        labels_batch.attention_mask.ne(1), model.IGNORE_LOSS_INDEX
    )
    return batch


def compute_metrics(eval_pred: EvalPrediction):
    logits = eval_pred.predictions
    labels = eval_pred.label_ids
    predictions = np.argmax(logits, axis=-1)
    valid_mask = labels != model.IGNORE_LOSS_INDEX
    frame_accuracy = (predictions[valid_mask] == labels[valid_mask]).mean()
    return {"frame_accuracy": frame_accuracy}

In [ ]:
training_args = TrainingArguments(
    output_dir="mms-300m-ForcedAligner-karaoke-ja-Latn",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=4,
    eval_accumulation_steps=1,
    gradient_checkpointing=True,
    bf16=True,
    tf32=True,
    learning_rate=2e-5,
    warmup_steps=0.05,
    eval_strategy="steps",
    eval_steps=0.1,
    save_steps=0.1,
    save_total_limit=5,
    metric_for_best_model="frame_accuracy",
    load_best_model_at_end=True,
    logging_steps=10,
    report_to="trackio",
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    processing_class=processor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

In [ ]:
trainer.train()

In [ ]:
processor.save_pretrained("mms-300m-ForcedAligner-karaoke-ja-Latn")
trainer.save_model()